In [17]:
import pandas as pd
from pathlib import Path

DATA_PATH = Path('data')

xlsx_path = DATA_PATH / 'Breakfast_at_the_Frat.xlsx'

TRANSACTION_SHEET = 'dh Transaction Data'
PRODUCT_SHEET     = 'dh Products Lookup'
STORE_SHEET       = 'dh Store Lookup'

transactions_df = pd.read_excel(xlsx_path, sheet_name=TRANSACTION_SHEET, header=1)
products_df     = pd.read_excel(xlsx_path, sheet_name=PRODUCT_SHEET,     header=1)
stores_df       = pd.read_excel(xlsx_path, sheet_name=STORE_SHEET,       header=1)

transactions_df['WEEK_END_DATE'] = pd.to_datetime(transactions_df['WEEK_END_DATE'])

out_dir = DATA_PATH / 'csv'
out_dir.mkdir(exist_ok=True)

transactions_df.to_csv(out_dir / 'transactions.csv', index=False)
products_df.to_csv(out_dir / 'products.csv', index=False)
stores_df.to_csv(out_dir / 'stores.csv', index=False)

for name, df in [('transactions', transactions_df),
                 ('products',     products_df),
                 ('stores',       stores_df)]:
    print(f'\n=== {name} | shape={df.shape} ===')
    print('columns:', df.columns.tolist())
    print('dtypes:')
    print(df.dtypes)
    print('missing:', df.isnull().sum().to_dict())
    print('head:')
    print(df.head(3))


=== transactions | shape=(524950, 12) ===
columns: ['WEEK_END_DATE', 'STORE_NUM', 'UPC', 'UNITS', 'VISITS', 'HHS', 'SPEND', 'PRICE', 'BASE_PRICE', 'FEATURE', 'DISPLAY', 'TPR_ONLY']
dtypes:
WEEK_END_DATE    datetime64[us]
STORE_NUM                 int64
UPC                       int64
UNITS                     int64
VISITS                    int64
HHS                       int64
SPEND                   float64
PRICE                   float64
BASE_PRICE              float64
FEATURE                   int64
DISPLAY                   int64
TPR_ONLY                  int64
dtype: object
missing: {'WEEK_END_DATE': 0, 'STORE_NUM': 0, 'UPC': 0, 'UNITS': 0, 'VISITS': 0, 'HHS': 0, 'SPEND': 0, 'PRICE': 23, 'BASE_PRICE': 185, 'FEATURE': 0, 'DISPLAY': 0, 'TPR_ONLY': 0}
head:
  WEEK_END_DATE  STORE_NUM         UPC  UNITS  VISITS  HHS  SPEND  PRICE  BASE_PRICE  FEATURE  DISPLAY  TPR_ONLY
0    2009-01-14        367  1111009477     13      13   13  18.07   1.39        1.57        0        0         1
1 

In [18]:
print('Unique stores  :', transactions_df['STORE_NUM'].nunique())
print('Unique UPCs    :', transactions_df['UPC'].nunique())
print('Date range     :', transactions_df['WEEK_END_DATE'].min(), '→', transactions_df['WEEK_END_DATE'].max())
print('Total weeks    :', transactions_df['WEEK_END_DATE'].nunique())

dup = transactions_df.duplicated(subset=['STORE_NUM', 'UPC', 'WEEK_END_DATE']).sum()
print('Duplicates on (store, upc, week):', dup)

print('\nProducts by CATEGORY:')
print(products_df['CATEGORY'].value_counts())

print('\nProducts by SUB_CATEGORY:')
print(products_df['SUB_CATEGORY'].value_counts())

print('\nStores by SEG_VALUE_NAME:')
print(stores_df['SEG_VALUE_NAME'].value_counts())

print('\nPRICE describe:')
print(transactions_df['PRICE'].describe(percentiles=[.01, .05, .5, .95, .99]))

print('\nUNITS describe:')
print(transactions_df['UNITS'].describe(percentiles=[.01, .05, .5, .95, .99]))

print('\nzero UNITS:', (transactions_df['UNITS'] == 0).sum())
print('zero/neg PRICE:', (transactions_df['PRICE'] <= 0).sum())

for col in ['FEATURE', 'DISPLAY', 'TPR_ONLY']:
    print(f'mean({col}) = {transactions_df[col].mean():.3f}')

Unique stores  : 77
Unique UPCs    : 55
Date range     : 2009-01-14 00:00:00 → 2012-01-04 00:00:00
Total weeks    : 156
Duplicates on (store, upc, week): 0

Products by CATEGORY:
CATEGORY
BAG SNACKS               15
COLD CEREAL              15
FROZEN PIZZA             15
ORAL HYGIENE PRODUCTS    13
Name: count, dtype: int64

Products by SUB_CATEGORY:
SUB_CATEGORY
PRETZELS                       15
PIZZA/PREMIUM                  15
MOUTHWASHES (ANTISEPTIC)        8
ALL FAMILY CEREAL               7
KIDS CEREAL                     5
MOUTHWASH/RINSES AND SPRAYS     5
ADULT CEREAL                    3
Name: count, dtype: int64

Stores by SEG_VALUE_NAME:
SEG_VALUE_NAME
MAINSTREAM    43
VALUE         19
UPSCALE       17
Name: count, dtype: int64

PRICE describe:
count    524927.000000
mean          3.382174
std           1.559303
min           0.000000
1%            0.980000
5%            1.250000
50%           2.990000
95%           6.590000
99%           7.490000
max          11.460000
Name

In [19]:
# 1) SPEND ≈ PRICE * UNITS?
check = transactions_df.assign(spend_calc=lambda d: d['PRICE'] * d['UNITS'])
diff = (check['SPEND'] - check['spend_calc']).abs()
print('SPEND vs PRICE*UNITS — max abs diff:', diff.max())
print('SPEND vs PRICE*UNITS — mean abs diff:', diff.mean())
print('Rows with diff > $0.50:', (diff > 0.5).sum())

# 2) PRICE vs BASE_PRICE
print('\nPRICE <= BASE_PRICE:', (transactions_df['PRICE'] <= transactions_df['BASE_PRICE']).mean())
print('Average discount depth (BASE - PRICE) / BASE:',
      ((transactions_df['BASE_PRICE'] - transactions_df['PRICE']) / transactions_df['BASE_PRICE']).mean())

# 3) Доля строк со скидкой (PRICE заметно ниже BASE_PRICE)
on_promo = (transactions_df['PRICE'] < transactions_df['BASE_PRICE'] - 0.01)
print('Share of rows with price below base:', on_promo.mean())

# 4) Комбинации промо-флагов
combo = transactions_df.groupby(['FEATURE', 'DISPLAY', 'TPR_ONLY']).size().reset_index(name='rows')
combo['share'] = combo['rows'] / combo['rows'].sum()
print('\nPromo flag combinations:')
print(combo.sort_values('rows', ascending=False))

SPEND vs PRICE*UNITS — max abs diff: 0.0
SPEND vs PRICE*UNITS — mean abs diff: 0.0
Rows with diff > $0.50: 0

PRICE <= BASE_PRICE: 0.9880845794837604
Average discount depth (BASE - PRICE) / BASE: 0.057198432323293
Share of rows with price below base: 0.25471568720830556

Promo flag combinations:
   FEATURE  DISPLAY  TPR_ONLY    rows     share
0        0        0         0  375564  0.715428
1        0        0         1   70734  0.134744
2        0        1         0   34401  0.065532
4        1        1         0   23414  0.044602
3        1        0         0   20837  0.039693


In [22]:
# === Cell 5b: исправленный мерж (без дублирования строк транзакций) ===

print('Before cleaning:', transactions_df.shape)

mask_clean = (
    transactions_df['PRICE'].notna() &
    transactions_df['BASE_PRICE'].notna() &
    (transactions_df['PRICE'] > 0) &
    (transactions_df['BASE_PRICE'] > 0) &
    (transactions_df['UNITS'] > 0)
)
df = transactions_df.loc[mask_clean].copy()
print('After cleaning :', df.shape)

# Фильтр на COLD CEREAL
cereal_upcs = products_df.loc[products_df['CATEGORY'] == 'COLD CEREAL', 'UPC'].tolist()
df = df[df['UPC'].isin(cereal_upcs)].copy()
print('After category filter:', df.shape)

# Готовим справочник магазинов — убираем дубли по STORE_ID
# Берём первое появление каждого STORE_ID (известный артефакт в датасете)
stores_clean = stores_df.drop_duplicates(subset='STORE_ID', keep='first').copy()
print(f'\nstores_df: {len(stores_df)} → {len(stores_clean)} after dedup')

# Мерж с products (one-to-one, проверено)
df = df.merge(
    products_df[['UPC', 'DESCRIPTION', 'MANUFACTURER', 'SUB_CATEGORY', 'PRODUCT_SIZE']],
    on='UPC', how='left', validate='many_to_one'
)

# Мерж со stores — теперь без дублей
df = df.merge(
    stores_clean[['STORE_ID', 'SEG_VALUE_NAME', 'MSA_CODE', 'SALES_AREA_SIZE_NUM', 'AVG_WEEKLY_BASKETS']]
        .rename(columns={'STORE_ID': 'STORE_NUM'}),
    on='STORE_NUM', how='left', validate='many_to_one'
)
print('After merges        :', df.shape)

# Финальные проверки
assert df['SEG_VALUE_NAME'].isna().sum() == 0, 'There are stores without segment'
print('\nmissing after merge:', df.isnull().sum().to_dict())
print('\nRows by store segment:')
print(df['SEG_VALUE_NAME'].value_counts(dropna=False))
print('\nDate coverage:')
print('weeks:', df['WEEK_END_DATE'].nunique(),
      ' | stores:', df['STORE_NUM'].nunique(),
      ' | UPCs:', df['UPC'].nunique())

Before cleaning: (524950, 12)
After cleaning : (524741, 12)
After category filter: (169677, 12)

stores_df: 79 → 77 after dedup
After merges        : (169677, 20)

missing after merge: {'WEEK_END_DATE': 0, 'STORE_NUM': 0, 'UPC': 0, 'UNITS': 0, 'VISITS': 0, 'HHS': 0, 'SPEND': 0, 'PRICE': 0, 'BASE_PRICE': 0, 'FEATURE': 0, 'DISPLAY': 0, 'TPR_ONLY': 0, 'DESCRIPTION': 0, 'MANUFACTURER': 0, 'SUB_CATEGORY': 0, 'PRODUCT_SIZE': 0, 'SEG_VALUE_NAME': 0, 'MSA_CODE': 0, 'SALES_AREA_SIZE_NUM': 0, 'AVG_WEEKLY_BASKETS': 0}

Rows by store segment:
SEG_VALUE_NAME
MAINSTREAM    95213
VALUE         41571
UPSCALE       32893
Name: count, dtype: int64

Date coverage:
weeks: 156  | stores: 77  | UPCs: 15
